In [2]:
import h5py
import torch as t
import numpy as np
import plotly.graph_objects as go
from jaxtyping import Float
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import Tensor

In [3]:
BEST_ACT_SET = {
    "arabic": "avg_final",
    "english": "avg",
    "german": "avg",
    "indonesian": "avg_final",
    "italian": "avg",
    "mixed": "mixed",
    "russian": "avg",
    "spanish": "avg",
    "thai": "avg",
}

In [4]:
def load_activations_and_labels(filepath_prefix : str, lang):
    """loads activations and corresponding labels for specified language from h5py file and returns tuple of dictionary of activations per layer and corresponding labels as torch tensor

    Args:
        filepath (str): path to h5py file
        lang (str): any supported language (english, german, indonesian)
    """
    filepath = f"{filepath_prefix}_{BEST_ACT_SET[lang]}.h5"
    with h5py.File(filepath, "r") as f:
        activations = f[lang]["activations"][:]  # shape: (N, n_layers, hidden_dim)
        labels = f[lang]["labels"][:]
            
    activations_by_layer = {
    layer_idx: t.from_numpy(activations[:, layer_idx, :].copy())
    for layer_idx in range(activations.shape[1])}
    
    return activations_by_layer, t.from_numpy(labels.copy())


In [5]:
LANGUAGES = ["english", "german", "spanish", "italian", "russian", "arabic", "indonesian", "thai"]

ACTS_AND_LABELS = {
    "train_acts": {},
    "train_labels": {},
    "test_acts": {},
    "test_labels": {},
    "nat_acts": {},
    "nat_labels": {}}

for lang in LANGUAGES:
    
    train_acts, train_labs = load_activations_and_labels("activations/train_completions", lang)
    
    test_acts, test_labs = load_activations_and_labels("activations/test_completions", lang)
    
    nat_acts, nat_labs = load_activations_and_labels("activations/nat_completions", lang)
    
    ACTS_AND_LABELS["train_acts"][lang] = train_acts
    ACTS_AND_LABELS["train_labels"][lang] = train_labs
    ACTS_AND_LABELS["test_acts"][lang] = test_acts
    ACTS_AND_LABELS["test_labels"][lang] = test_labs
    ACTS_AND_LABELS["nat_acts"][lang] = nat_acts
    ACTS_AND_LABELS["nat_labels"][lang] = nat_labs

In [6]:
def make_mixed_dataset(acts_dict, labels_dict, n_total, languages, pair_based=True, seed=42):
    rng = np.random.default_rng(seed)
    n_langs = len(languages)
    
    if pair_based:
        pairs_total = n_total // 2
        base_pairs = pairs_total // n_langs
        remainder = pairs_total % n_langs
        extras = rng.choice(n_langs, size=remainder, replace=False)
        pairs_per_lang = [base_pairs + (1 if i in extras else 0) for i in range(n_langs)]
    else:
        base = n_total // n_langs
        remainder = n_total % n_langs
        extras = rng.choice(n_langs, size=remainder, replace=False)
        per_lang = [base + (1 if i in extras else 0) for i in range(n_langs)]

    layers = list(acts_dict[languages[0]].keys())
    
    all_acts = {layer: [] for layer in layers}
    all_labels = []

    for i, lang in enumerate(languages):
        labels = labels_dict[lang]
        n_samples = len(labels)

        if pair_based:
            n_pairs = n_samples // 2
            selected_pairs = rng.choice(n_pairs, size=pairs_per_lang[i], replace=False)
            selected_idx = np.concatenate([[2*j, 2*j+1] for j in selected_pairs])
        else:
            selected_idx = rng.choice(n_samples, size=per_lang[i], replace=False)

        for layer in layers:
            all_acts[layer].append(acts_dict[lang][layer][selected_idx])
        all_labels.append(labels[selected_idx])

    mixed_labels = t.cat(all_labels, dim=0)
    shuffle_idx = t.randperm(len(mixed_labels), generator=t.Generator().manual_seed(seed))
    
    mixed_acts = {}
    for layer in layers:
        mixed_acts[layer] = t.cat(all_acts[layer], dim=0)[shuffle_idx]
    
    return mixed_acts, mixed_labels[shuffle_idx]


# Build mixed datasets
ACTS_AND_LABELS["train_acts"]["mixed"], ACTS_AND_LABELS["train_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"],
        n_total=600, languages=LANGUAGES, pair_based=True)

ACTS_AND_LABELS["test_acts"]["mixed"], ACTS_AND_LABELS["test_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["test_acts"], ACTS_AND_LABELS["test_labels"],
        n_total=600, languages=LANGUAGES, pair_based=True)

ACTS_AND_LABELS["nat_acts"]["mixed"], ACTS_AND_LABELS["nat_labels"]["mixed"] = make_mixed_dataset(
        ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"],
        n_total=250, languages=LANGUAGES, pair_based=False)

LANGUAGES.append("mixed")

In [7]:
class MMProbe(t.nn.Module):
    def __init__(
        self,
        direction: Float[Tensor, " d_model"]
    ):
        super().__init__()
        self.direction = t.nn.Parameter(direction, requires_grad=False)

    def forward(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        
        return t.sigmoid(x @ self.direction)

    def pred(self, x: Float[Tensor, "n d_model"]) -> Float[Tensor, " n"]:
        return self(x, ).round()

    @staticmethod
    def from_data(
        acts: Float[Tensor, "n d_model"],
        labels: Float[Tensor, " n"],
        device: str = "cpu",
    ) -> "MMProbe":
        acts, labels = acts.to(device), labels.to(device)
        pos_acts = acts[labels == 1]
        neg_acts = acts[labels == 0]
        pos_mean = pos_acts.mean(0)
        neg_mean = neg_acts.mean(0)
        direction = pos_mean - neg_mean 

        centered = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.t() @ centered / acts.shape[0]

        return MMProbe(direction).to(device)

In [8]:
class LRProbe(t.nn.Module):
    """Logistic Regression probe (sklearn backend, torch wrapper)."""
    def __init__(self, d_in, scaler_mean=None, scaler_scale=None):
        super().__init__()
        self.net = t.nn.Sequential(t.nn.Linear(d_in, 1, bias=False), t.nn.Sigmoid())
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def _normalize(self, x):
        if self.scaler_mean is not None and self.scaler_scale is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x):
        return self.net(self._normalize(x)).squeeze(-1)

    def pred(self, x):
        return self(x).round()
    
    @property
    def direction(self) -> Float[Tensor, " d_model"]:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(acts, labels, C=0.1, device="cpu"):
        X = acts.cpu().float().numpy()
        y = labels.cpu().float().numpy()

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000)
        lr_model.fit(X_scaled, y)

        scaler_mean = t.tensor(scaler.mean_, dtype=t.float32)
        scaler_scale = t.tensor(scaler.scale_, dtype=t.float32)
        probe = LRProbe(acts.shape[-1], scaler_mean=scaler_mean, scaler_scale=scaler_scale).to(device)
        probe.net[0].weight.data[0] = t.tensor(lr_model.coef_[0], dtype=t.float32).to(device)
        return probe

In [9]:
BEST_LAYERS = {
    "arabic": {"mm": 15, "lr": 18},
    "english": {"mm": 24, "lr": 22},
    "german": {"mm": 24, "lr": 23},
    "indonesian": {"mm": 17, "lr": 23},
    "italian": {"mm": 17, "lr": 23},
    "mixed": {"mm": 27, "lr": 23},
    "russian": {"mm": 25, "lr": 24},
    "spanish": {"mm": 17, "lr": 21},
    "thai": {"mm": 15, "lr": 16},
}

In [19]:
def compute_generalization_matrix_best_layer(
    train_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
) :
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    probes = {}
    probe_acronym = "mm" if probe_cls == MMProbe else "lr"
    for lang in LANGUAGES:
        train_act = train_acts[lang][BEST_LAYERS[lang][probe_acronym]][:600,:]
        train_lab = train_labels[lang][:600]
        probe = probe_cls.from_data(train_act, train_lab)
        
        probes[lang] = probe
        
    roc_auc = t.zeros(len(dataset_names), len(dataset_names)).float()
    
    for i, lang_i in enumerate(dataset_names):
        probe = probes[lang_i]
        
        for j, lang_j in enumerate(dataset_names):
            test_act = test_acts[lang_j]
            test_lab = test_labels[lang_j]
            test_preds = probe(test_act[BEST_LAYERS[lang_i][probe_acronym]])
            test_auc = roc_auc_score(test_lab.detach().numpy(), test_preds.detach().numpy())
            
            roc_auc[i,j] = float(test_auc)
            
            
    return roc_auc, probes



mm_matrix_nat, mm_probes = compute_generalization_matrix_best_layer(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, MMProbe)

lr_matrix_nat, lr_probes = compute_generalization_matrix_best_layer(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, LRProbe)


# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=["Difference-in-Means Probe", "Logistic Regression Probe"], horizontal_spacing=0.15)

for idx, (matrix, name) in enumerate([(mm_matrix_nat, "mm"), (lr_matrix_nat, "lr")]):
    text_vals = [[f"{matrix[i, j]:.2f}" for j in range(len(LANGUAGES))] for i in range(len(LANGUAGES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            y=[f"{lang.capitalize()} (L{BEST_LAYERS[lang][name]})" for lang in LANGUAGES],
            x=[lang.capitalize() for lang in LANGUAGES],
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdBu",
            zmin=0.4,
            zmax=1.0,
            showscale=(idx == 1),
            textfont=dict(size=11)
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Training Dataset Language (Best Layer)" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test Dataset Language", row=1, col=idx + 1)

fig.update_layout(title=dict(
        text="Cross-Language Probe Generalization (ROC-AUC-Score, Language-Specific Best Layer)",
        x=0.5,
        xanchor="center", font=dict(size=14)
    ),font=dict(size=11),height=400, width=900)
for ann in fig["layout"]["annotations"]:
    ann["font"] = dict(size=12)
    
#plot_path = "figures/heat_map_best_layers.pdf"
fig.write_image("figures/rq2_heat_map_best_layer.png", scale = 3)
fig.show()

# Cosine similarity between probe directions
mm_directions = {lang: mm_probes[lang].direction for lang in LANGUAGES}
lr_directions = {lang: lr_probes[lang].direction for lang in LANGUAGES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions), ("LR", lr_directions)]:
    print(f"\n% {probe_name} Probe")
    n = len(LANGUAGES)
    
    # Precompute normalized directions
    normed = {lang: directions[lang] / directions[lang].norm() for lang in LANGUAGES}
    
    # Build similarity matrix
    sims = {}
    for i, n1 in enumerate(LANGUAGES):
        for j, n2 in enumerate(LANGUAGES):
            sims[(n1, n2)] = (normed[n1] @ normed[n2]).item()
    
    # LaTeX output
    col_fmt = "l" + "c" * n
    print(f"\\begin{{tabular}}{{{col_fmt}}}")
    print("\\toprule")
    print(" & " + " & ".join(LANGUAGES) + " \\\\")
    print("\\midrule")
    for n1 in LANGUAGES:
        row_vals = []
        for n2 in LANGUAGES:
            val = sims[(n1, n2)]
            if n1 == n2:
                row_vals.append("1.00")
            else:
                row_vals.append(f"{val:.2f}")
        print(f"{n1} & " + " & ".join(row_vals) + " \\\\")
    print("\\bottomrule")
    print(f"\\end{{tabular}}")
    
normed_mm = {lang: mm_directions[lang] / mm_directions[lang].norm() for lang in LANGUAGES}
normed_lr = {lang: lr_directions[lang] / lr_directions[lang].norm() for lang in LANGUAGES}

print("Cosine Similarity of Probe Directions by Language")
for lang in LANGUAGES:
    sim = (normed_mm[lang] @ normed_lr[lang]).item()
    print(f"{lang.capitalize()}:    {sim:.2f}")
    


Pairwise cosine similarity between probe directions:

% MM Probe
\begin{tabular}{lccccccccc}
\toprule
 & english & german & spanish & italian & russian & arabic & indonesian & thai & mixed \\
\midrule
english & 1.00 & 0.88 & 0.53 & 0.47 & 0.60 & 0.24 & 0.33 & 0.27 & 0.63 \\
german & 0.88 & 1.00 & 0.56 & 0.49 & 0.62 & 0.24 & 0.33 & 0.29 & 0.67 \\
spanish & 0.53 & 0.56 & 1.00 & 0.81 & 0.48 & 0.35 & 0.54 & 0.42 & 0.50 \\
italian & 0.47 & 0.49 & 0.81 & 1.00 & 0.43 & 0.30 & 0.50 & 0.35 & 0.45 \\
russian & 0.60 & 0.62 & 0.48 & 0.43 & 1.00 & 0.23 & 0.30 & 0.27 & 0.63 \\
arabic & 0.24 & 0.24 & 0.35 & 0.30 & 0.23 & 1.00 & 0.42 & 0.24 & 0.30 \\
indonesian & 0.33 & 0.33 & 0.54 & 0.50 & 0.30 & 0.42 & 1.00 & 0.32 & 0.40 \\
thai & 0.27 & 0.29 & 0.42 & 0.35 & 0.27 & 0.24 & 0.32 & 1.00 & 0.34 \\
mixed & 0.63 & 0.67 & 0.50 & 0.45 & 0.63 & 0.30 & 0.40 & 0.34 & 1.00 \\
\bottomrule
\end{tabular}

% LR Probe
\begin{tabular}{lccccccccc}
\toprule
 & english & german & spanish & italian & russian & arabic & 

In [11]:
BEST_UNIVERSAL_LAYER = {
    "mm": 25,
    "lr": 23,
}

In [18]:
def compute_generalization_matrix(
    train_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    train_labels: dict[str, Float[Tensor, " n"]],
    test_acts: dict[str, dict[int, Float[Tensor, "n d"]]],
    test_labels: dict[str, Float[Tensor, " n"]],
    dataset_names: list[str],
    probe_cls: type,
    layer: int
) :
    """
    Compute a generalization matrix: entry (i, j) is the test accuracy of a probe trained on dataset i
    and evaluated on dataset j.

    Args:
        train_acts, train_labels: Training data per dataset.
        test_acts, test_labels: Test data per dataset.
        dataset_names: Names of datasets (determines matrix ordering).
        probe_cls: Probe class to use (MMProbe or LRProbe), must have from_data and pred methods.

    Returns:
        Tensor of shape [n_datasets, n_datasets] with accuracy values.
    """
    probes = {}
    probe_acronym = "mm" if probe_cls == MMProbe else "lr"
    for lang in LANGUAGES:
        train_act = train_acts[lang][layer][:600,:]
        train_lab = train_labels[lang][:600]
        probe = probe_cls.from_data(train_act, train_lab)
        
        probes[lang] = probe
        
    roc_auc = t.zeros(len(dataset_names), len(dataset_names)).float()
    
    for i, lang_i in enumerate(dataset_names):
        probe = probes[lang_i]
        
        for j, lang_j in enumerate(dataset_names):
            test_act = test_acts[lang_j]
            test_lab = test_labels[lang_j]
            test_preds = probe(test_act[layer])
            test_auc = roc_auc_score(test_lab.detach().numpy(), test_preds.detach().numpy())
            
            roc_auc[i,j] = float(test_auc)
            
            
    return roc_auc, probes
        

mm_matrix_nat, mm_probes = compute_generalization_matrix(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, MMProbe, BEST_UNIVERSAL_LAYER["mm"])

lr_matrix_nat, lr_probes = compute_generalization_matrix(ACTS_AND_LABELS["train_acts"], ACTS_AND_LABELS["train_labels"], ACTS_AND_LABELS["nat_acts"], ACTS_AND_LABELS["nat_labels"], LANGUAGES, LRProbe, BEST_UNIVERSAL_LAYER["lr"])


# Heatmap visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"Difference-in-Means Probe (Layer: {BEST_UNIVERSAL_LAYER["mm"]})", f"Logistic Regression Probe (Layer: {BEST_UNIVERSAL_LAYER["lr"]})"], horizontal_spacing=0.15, )

for idx, (matrix, name) in enumerate([(mm_matrix_nat, "MM-probe"), (lr_matrix_nat, "LR-probe")]):
    text_vals = [[f"{matrix[i, j]:.3f}" for j in range(len(LANGUAGES))] for i in range(len(LANGUAGES))]
    fig.add_trace(
        go.Heatmap(
            z=matrix.numpy(),
            x=[lang.capitalize() for lang in LANGUAGES],
            y=[lang.capitalize() for lang in LANGUAGES],
            text=text_vals,
            texttemplate="%{text}",
            colorscale="RdBu",
            zmin=0.4,
            zmax=1.0,
            showscale=(idx == 1),
        ),
        row=1,
        col=idx + 1,
    )
    fig.update_yaxes(title_text="Training Dataset Language" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Test Dataset Language", row=1, col=idx + 1)

fig.update_layout(title=dict(
        text="Cross-Language Probe Generalization (ROC-AUC-Score, Fixed Layer)",
        x=0.5,
        xanchor="center", font=dict(size=14)
    ),font=dict(size=11),height=400, width=900)
for ann in fig["layout"]["annotations"]:
    ann["font"] = dict(size=12)
    
fig.write_image("figures/rq2_heat_map_universal_layer.png", scale=3)
fig.show()      

# Cosine similarity between probe directions
mm_directions = {lang: mm_probes[lang].direction for lang in LANGUAGES}
lr_directions = {lang: lr_probes[lang].direction for lang in LANGUAGES}

print("\nPairwise cosine similarity between probe directions:")
for probe_name, directions in [("MM", mm_directions), ("LR", lr_directions)]:
    print(f"\n% {probe_name} Probe")
    n = len(LANGUAGES)
    
    # Precompute normalized directions
    normed = {lang: directions[lang] / directions[lang].norm() for lang in LANGUAGES}
    
    # Build similarity matrix
    sims = {}
    for i, n1 in enumerate(LANGUAGES):
        for j, n2 in enumerate(LANGUAGES):
            sims[(n1, n2)] = (normed[n1] @ normed[n2]).item()
    
    # LaTeX output
    col_fmt = "l" + "c" * n
    print(f"\\begin{{tabular}}{{{col_fmt}}}")
    print("\\toprule")
    print(" & " + " & ".join(LANGUAGES) + " \\\\")
    print("\\midrule")
    for n1 in LANGUAGES:
        row_vals = []
        for n2 in LANGUAGES:
            val = sims[(n1, n2)]
            if n1 == n2:
                row_vals.append("1.00")
            else:
                row_vals.append(f"{val:.2f}")
        print(f"{n1} & " + " & ".join(row_vals) + " \\\\")
    print("\\bottomrule")
    print(f"\\end{{tabular}}")
    
normed_mm = {lang: mm_directions[lang] / mm_directions[lang].norm() for lang in LANGUAGES}
normed_lr = {lang: lr_directions[lang] / lr_directions[lang].norm() for lang in LANGUAGES}

print("Cosine Similarity of Probe Directions by Language")
for lang in LANGUAGES:
    sim = (normed_mm[lang] @ normed_lr[lang]).item()
    print(f"{lang.capitalize()}:    {sim:.2f}")
    


Pairwise cosine similarity between probe directions:

% MM Probe
\begin{tabular}{lccccccccc}
\toprule
 & english & german & spanish & italian & russian & arabic & indonesian & thai & mixed \\
\midrule
english & 1.00 & 0.88 & 0.85 & 0.78 & 0.65 & 0.50 & 0.53 & 0.47 & 0.88 \\
german & 0.88 & 1.00 & 0.89 & 0.79 & 0.67 & 0.56 & 0.56 & 0.52 & 0.90 \\
spanish & 0.85 & 0.89 & 1.00 & 0.84 & 0.70 & 0.55 & 0.56 & 0.56 & 0.92 \\
italian & 0.78 & 0.79 & 0.84 & 1.00 & 0.64 & 0.52 & 0.55 & 0.52 & 0.88 \\
russian & 0.65 & 0.67 & 0.70 & 0.64 & 1.00 & 0.42 & 0.41 & 0.44 & 0.77 \\
arabic & 0.50 & 0.56 & 0.55 & 0.52 & 0.42 & 1.00 & 0.66 & 0.31 & 0.70 \\
indonesian & 0.53 & 0.56 & 0.56 & 0.55 & 0.41 & 0.66 & 1.00 & 0.35 & 0.72 \\
thai & 0.47 & 0.52 & 0.56 & 0.52 & 0.44 & 0.31 & 0.35 & 1.00 & 0.66 \\
mixed & 0.88 & 0.90 & 0.92 & 0.88 & 0.77 & 0.70 & 0.72 & 0.66 & 1.00 \\
\bottomrule
\end{tabular}

% LR Probe
\begin{tabular}{lccccccccc}
\toprule
 & english & german & spanish & italian & russian & arabic & 